[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Concurrency and WAL &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations, their year of readings and the
`counters` table, in the default rollback journal mode, and defines `connect` and `attempt` as the
notebook did. Every task opens and closes connections of its own, and the tasks that change the
journal mode use databases of their own, so the tasks do not depend on one another. The last cell
removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
for leftover in SCRATCH.glob("*.db*"):
    leftover.unlink()
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
NEW_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"
COUNT_2026 = "SELECT COUNT(*) FROM readings WHERE hour >= '2026'"


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE counters (name TEXT PRIMARY KEY, value INTEGER NOT NULL);
    INSERT INTO counters VALUES ('hours loaded', 8760);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany(NEW_READING, ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def connect(path=DATABASE, timeout=0.1):
    """A connection that writes its own BEGIN and COMMIT, and waits `timeout` seconds for a lock."""
    return sqlite3.connect(path, autocommit=True, timeout=timeout)


def attempt(conn, sql, parameters=()):
    """Run one statement, and say what happened: its rows or its error, at once or after waiting for a lock."""
    started = time.perf_counter()
    try:
        cursor = conn.execute(sql, parameters)
        outcome = f"ran {cursor.fetchall()}" if cursor.description else "ran"
    except sqlite3.OperationalError as error:
        outcome = f"OperationalError: {error}"
    return outcome + (", after waiting" if time.perf_counter() - started > 0.05 else ", at once")


print("built", DATABASE)


built scratch/stations.db


**1.** Reading beside an open write, in the rollback journal mode.


In [2]:
writer, reader = connect(), connect()
print("writer BEGIN:  ", attempt(writer, "BEGIN"))
print("writer INSERT: ", attempt(writer, NEW_READING, (ids["Oslo"], "2026-02-01T00:00", -1.2)))
print("reader SELECT: ", attempt(reader, COUNT_2026))
print("writer COMMIT: ", attempt(writer, "COMMIT"))
print("reader SELECT: ", attempt(reader, COUNT_2026))
writer.close()
reader.close()


writer BEGIN:   ran, at once
writer INSERT:  ran, at once
reader SELECT:  ran [(0,)], at once
writer COMMIT:  ran, at once
reader SELECT:  ran [(1,)], at once


The writer's insert took a RESERVED lock, which lets other connections go on reading, so the reader's
query ran at once, and counted no readings from 2026: the uncommitted row exists only inside the
writer's transaction. After the commit, the reader counted it.


**2.** A longer busy timeout.


In [3]:
holder, waiter = connect(), connect()
holder.execute("BEGIN IMMEDIATE")


def time_blocked_insert():
    """Seconds an insert waits for the held lock before it fails."""
    started = time.perf_counter()
    try:
        waiter.execute(NEW_READING, (ids["Bergen"], "2026-02-01T00:00", 3.1))
    except sqlite3.OperationalError:
        pass
    return time.perf_counter() - started


print("with timeout=0.1, waited under 0.3 seconds:", time_blocked_insert() < 0.3)
waiter.execute("PRAGMA busy_timeout = 400")
print("with busy_timeout = 400, waited over 0.3 seconds:", time_blocked_insert() > 0.3)
holder.execute("ROLLBACK")
holder.close()
waiter.close()


with timeout=0.1, waited under 0.3 seconds: True
with busy_timeout = 400, waited over 0.3 seconds: True


`PRAGMA busy_timeout` replaced the tenth of a second the connection was opened with, and the same
blocked insert waited four times as long before it failed. The setting belongs to the connection, and
lasts until it is changed or the connection closes.


**3.** A snapshot in WAL mode.


In [4]:
wal_path = SCRATCH / "snapshot.db"
shutil.copy(DATABASE, wal_path)
setup = connect(wal_path)
setup.execute("PRAGMA journal_mode = WAL")
setup.close()

TROMSO_2026 = "SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour >= '2026'"
reader, writer = connect(wal_path), connect(wal_path)
reader.execute("BEGIN")
print("reader, before:", reader.execute(TROMSO_2026, (ids["Tromso"],)).fetchone()[0])
for hour in range(3):
    writer.execute(NEW_READING, (ids["Tromso"], f"2026-02-01T{hour:02d}:00", -6.0))
print("reader, same transaction:", reader.execute(TROMSO_2026, (ids["Tromso"],)).fetchone()[0])
reader.execute("COMMIT")
print("reader, new transaction:", reader.execute(TROMSO_2026, (ids["Tromso"],)).fetchone()[0])
reader.close()
writer.close()


reader, before: 0
reader, same transaction: 0
reader, new transaction: 3


The three inserts committed at once while the reader's transaction was open, since in WAL mode a
commit does not wait for readers. The reader kept its snapshot, with none of Tromso's readings from
2026, until it ended its transaction, and its next query began a new snapshot with all three.


**4.** A journal mode that belongs to the file.


In [5]:
sticky_path = SCRATCH / "sticky.db"
first = connect(sticky_path)
first.execute("CREATE TABLE notes (note TEXT)")
print("set:", first.execute("PRAGMA journal_mode = WAL").fetchone()[0])
first.close()

later = connect(sticky_path)
print("a connection opened later:", later.execute("PRAGMA journal_mode").fetchone()[0])
later.close()


set: wal
a connection opened later: wal


The first connection closed, and the file kept the setting: the next connection found it in WAL mode
without asking. `synchronous` and `busy_timeout`, by contrast, belong to the connection and are gone
when it closes.


**5.** Increments that cannot be lost.


In [6]:
holding = threading.Event()


def increment(conn, pause=0.0, signal=None):
    """Read the counter and write it back plus one, inside one BEGIN IMMEDIATE transaction."""
    conn.execute("BEGIN IMMEDIATE")
    if signal:
        signal.set()
    value = conn.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
    time.sleep(pause)
    conn.execute("UPDATE counters SET value = ? WHERE name = 'hours loaded'", (value + 1,))
    conn.execute("COMMIT")


def increment_in_a_thread():
    """One increment that holds its lock for a moment, on the thread's own connection."""
    conn = connect(timeout=5.0)
    increment(conn, pause=0.2, signal=holding)
    conn.close()


main = connect(timeout=5.0)
before = main.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
thread = threading.Thread(target=increment_in_a_thread)
thread.start()
holding.wait()
increment(main)
thread.join()
print("the counter rose by:", main.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0] - before)
main.close()


the counter rose by: 2


The thread took the write lock, read the counter and paused. The main thread's `BEGIN IMMEDIATE`
waited for the thread's commit before it read anything, so it read the value the thread had written,
and both increments landed. Without `IMMEDIATE`, both would have read the same value.


**6.** Back to the rollback journal.


In [7]:
back_path = SCRATCH / "back.db"
conn = connect(back_path)
conn.execute("PRAGMA journal_mode = WAL")
conn.execute("CREATE TABLE notes (note TEXT)")
conn.execute("INSERT INTO notes VALUES ('written in WAL mode')")
print("while open:", sorted(path.name for path in SCRATCH.glob("back.db*")))

print("journal_mode:", conn.execute("PRAGMA journal_mode = DELETE").fetchone()[0])
conn.close()
print("after closing:", sorted(path.name for path in SCRATCH.glob("back.db*")))


while open: ['back.db', 'back.db-shm', 'back.db-wal']
journal_mode: delete
after closing: ['back.db']


In WAL mode, the open database had its `-wal` and `-shm` files beside it. Changing back to `DELETE`
checkpointed the log into the database file, and once the connection closed, only the database file
was left, complete by itself.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Concurrency and WAL](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/17-concurrency-and-wal.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
